# Assignment 12: Comprehensive CLIP Problem (100 points)

**Unit 10: Computer Vision & Generative AI | AI 520**

**THIS IS A FULL ROUND 2-STYLE EXAM PROBLEM.**

You will build a complete CLIP system from scratch: Vision Transformer image encoder,
Transformer text encoder, contrastive learning with InfoNCE, and zero-shot classification.
This mirrors the 2025 USAAIO Round 2 Problem 3 (100 points).

**Time target**: 90 minutes

**Format**: USAAIO Round 2 Official

In [ ]:
# DO NOT CHANGE THIS CELL
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math

---
**WARNING**: Do not import any additional libraries. All implementations must use only `torch`, `numpy`, and `math`. Do not use `torchvision`, `transformers`, or any external packages.

---

## Background

CLIP (Contrastive Language-Image Pre-training) learns a joint embedding space for images and text.
Given a batch of $N$ image-text pairs, CLIP trains two encoders to maximize the cosine similarity
of matched pairs while minimizing similarity of mismatched pairs.

The system consists of:
1. **Image encoder** (Vision Transformer): maps images to embedding vectors
2. **Text encoder** (Transformer): maps text to embedding vectors
3. **Contrastive loss** (InfoNCE): symmetric cross-entropy on the similarity matrix
4. **Zero-shot classification**: match image embeddings against text class descriptions

## Part 1: Patch Embedding for ViT (7 points)

Split a 32x32 RGB image into 4x4 patches and project each to dimension $D=128$.

- Number of patches: $(32/4)^2 = 64$
- Each patch: $4 \times 4 \times 3 = 48$ values
- Use `nn.Conv2d(3, 128, kernel_size=4, stride=4)` for efficient implementation

**Input**: `(B, 3, 32, 32)` images
**Output**: `(B, 64, 128)` patch embeddings

In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_ch=3, embed_dim=128):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        ### WRITE YOUR SOLUTION HERE ###


    def forward(self, x):
        # x: (B, 3, 32, 32) -> (B, 64, 128)
        ### WRITE YOUR SOLUTION HERE ###


        """ END OF THIS PART """

## Part 2: Vision Transformer Image Encoder (10 points)

Build the image encoder:
1. Patch embedding -> `(B, 64, 128)`
2. Prepend learnable `[CLS]` token -> `(B, 65, 128)`
3. Add learnable position embeddings -> `(B, 65, 128)`
4. Apply `num_layers=4` transformer encoder layers (use `nn.TransformerEncoderLayer`)
5. Extract `[CLS]` token output -> `(B, 128)`

For the transformer layers, use `d_model=128, nhead=4, dim_feedforward=256, batch_first=True`.

In [ ]:
class VisionEncoder(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_ch=3, embed_dim=128, num_heads=4, num_layers=4):
        super().__init__()
        ### WRITE YOUR SOLUTION HERE ###


    def forward(self, images):
        # images: (B, 3, 32, 32) -> (B, 128)
        ### WRITE YOUR SOLUTION HERE ###


        """ END OF THIS PART """

## Part 3: Text Encoder (7 points)

Build the text encoder:
1. Token embedding: `nn.Embedding(vocab_size, embed_dim)` -> `(B, L, 128)`
2. Add learnable position embeddings -> `(B, L, 128)`
3. Apply `num_layers=2` transformer encoder layers
4. Take the mean over sequence positions -> `(B, 128)`

Use `d_model=128, nhead=4, dim_feedforward=256, batch_first=True`.

In [ ]:
class TextEncoder(nn.Module):
    def __init__(self, vocab_size=1000, max_len=32, embed_dim=128, num_heads=4, num_layers=2):
        super().__init__()
        ### WRITE YOUR SOLUTION HERE ###


    def forward(self, token_ids):
        # token_ids: (B, L) long tensor -> (B, 128)
        ### WRITE YOUR SOLUTION HERE ###


        """ END OF THIS PART """

## Part 4: Projection Heads (5 points)

Project both encoder outputs to a shared embedding space of dimension $D_{proj}=64$.

Each projection: `Linear(128, 64, bias=False)` followed by L2 normalization.

In [ ]:
class Projector(nn.Module):
    def __init__(self, in_dim=128, out_dim=64):
        super().__init__()
        ### WRITE YOUR SOLUTION HERE ###


    def forward(self, x):
        # x: (B, 128) -> project -> L2 normalize -> (B, 64)
        ### WRITE YOUR SOLUTION HERE ###


        """ END OF THIS PART """

## Part 5: Cosine Similarity Matrix (5 points)

Compute the $N \times N$ cosine similarity matrix between L2-normalized image and text embeddings.

Since embeddings are L2-normalized: $\cos(v_i, t_j) = v_i^T t_j$

In [ ]:
def compute_similarity(img_emb, txt_emb):
    """
    img_emb: (N, D) L2-normalized
    txt_emb: (N, D) L2-normalized
    Returns: (N, N) cosine similarity matrix
    """
    ### WRITE YOUR SOLUTION HERE ###


    """ END OF THIS PART """

## Part 6: Temperature-Scaled Logits (5 points)

Scale similarities by learnable temperature $\tau$ (stored as $\log(1/\tau)$):

$$\text{logits} = \frac{S}{\tau} = S \cdot \exp(\log(1/\tau))$$

In [ ]:
def scale_logits(similarity, log_inv_tau):
    """
    similarity: (N, N)
    log_inv_tau: scalar parameter
    Returns: (N, N) scaled logits
    """
    ### WRITE YOUR SOLUTION HERE ###


    """ END OF THIS PART """

## Part 7: Symmetric InfoNCE Loss (10 points)

Compute the symmetric contrastive loss:

$$\mathcal{L} = \frac{1}{2}\left[\text{CE}(\text{logits}, \text{labels}) + \text{CE}(\text{logits}^T, \text{labels})\right]$$

where $\text{labels} = [0, 1, 2, \ldots, N-1]$ (diagonal = positive pairs).

In [ ]:
def symmetric_infonce(logits):
    """
    logits: (N, N) temperature-scaled similarity matrix
    Returns: scalar symmetric InfoNCE loss
    """
    ### WRITE YOUR SOLUTION HERE ###


    """ END OF THIS PART """

## Part 8: Full CLIP Model Assembly (8 points)

Assemble all components into a single CLIP model.

In [ ]:
class CLIP(nn.Module):
    def __init__(self, vocab_size=1000, img_size=32, patch_size=4, embed_dim=128, proj_dim=64, tau_init=0.07):
        super().__init__()
        ### WRITE YOUR SOLUTION HERE ###
        # Components: vision_encoder, text_encoder, img_proj, txt_proj, log_inv_tau


    def forward(self, images, token_ids):
        """
        images: (N, 3, 32, 32)
        token_ids: (N, L)
        Returns: (logits, img_emb, txt_emb)
          logits: (N, N) scaled similarity matrix
          img_emb: (N, proj_dim) L2-normalized
          txt_emb: (N, proj_dim) L2-normalized
        """
        ### WRITE YOUR SOLUTION HERE ###


        """ END OF THIS PART """

## Part 9: Training Step (8 points)

Implement one training step: forward pass, compute loss, backward pass, optimizer step.

In [ ]:
def train_step(model, images, token_ids, optimizer):
    """
    One CLIP training step.
    Returns: dict with 'loss', 'tau' (current temperature)
    """
    ### WRITE YOUR SOLUTION HERE ###



    """ END OF THIS PART """

## Part 10: Zero-Shot Classification (10 points)

Given a trained CLIP model, classify images by comparing their embeddings
with embeddings of text descriptions for each class.

1. Encode all class descriptions: $t_k = \text{TextEncoder}(\text{"a photo of a } c_k\text{"})$
2. Encode the query image: $v = \text{ImageEncoder}(x)$
3. Predict: $\hat{y} = \arg\max_k \cos(v, t_k)$

In [ ]:
@torch.no_grad()
def zero_shot_predict(model, images, class_token_ids):
    """
    Zero-shot classification.
    images: (B, 3, 32, 32)
    class_token_ids: (K, L) token IDs for K class descriptions
    Returns: (predictions, probabilities)
      predictions: (B,) predicted class indices
      probabilities: (B, K) softmax probabilities
    """
    ### WRITE YOUR SOLUTION HERE ###



    """ END OF THIS PART """

## Part 11: Retrieval Metrics (8 points)

Compute Recall@K for image-to-text retrieval.

$$\text{Recall@K} = \frac{1}{N}\sum_{i=1}^{N} \mathbb{1}[\text{correct text}_i \in \text{top-K results for image}_i]$$

In [ ]:
@torch.no_grad()
def recall_at_k(img_emb, txt_emb, k=5):
    """
    Image-to-text Recall@K.
    img_emb: (N, D) L2-normalized image embeddings
    txt_emb: (N, D) L2-normalized text embeddings
    Assumes image i matches text i.
    Returns: scalar Recall@K
    """
    ### WRITE YOUR SOLUTION HERE ###



    """ END OF THIS PART """

## Part 12: Embedding Space Visualization Prep (5 points)

Compute a 2D t-SNE-like projection for visualization. Since we can't import sklearn,
use PCA (top-2 eigenvectors of the centered embedding matrix) instead.

**Input**: `(N, D)` embeddings
**Output**: `(N, 2)` projected coordinates

In [ ]:
@torch.no_grad()
def pca_2d(embeddings):
    """
    Project embeddings to 2D via PCA.
    embeddings: (N, D)
    Returns: (N, 2) projected coordinates
    """
    ### WRITE YOUR SOLUTION HERE ###



    """ END OF THIS PART """

## Part 13: Gradient Analysis (5 points)

Compute the gradient of the CLIP loss with respect to the temperature parameter.

Given a logits matrix and the current temperature, compute $\frac{\partial \mathcal{L}}{\partial \tau}$ analytically.

**Hint**: Since logits $= S / \tau$, we have $\frac{\partial \text{logits}}{\partial \tau} = -S/\tau^2 = -\text{logits}/\tau$.

In [ ]:
def temperature_gradient(model, images, token_ids):
    """
    Compute the gradient of CLIP loss w.r.t. the temperature parameter.
    Returns: scalar gradient value
    """
    ### WRITE YOUR SOLUTION HERE ###



    """ END OF THIS PART """

## Part 14: Full Pipeline Verification (7 points)

Create a complete CLIP model, run a training step, and perform zero-shot classification.
Verify all shapes are correct.

In [ ]:
# DO NOT CHANGE THIS CELL
# Create model
model = CLIP(vocab_size=100, img_size=32, patch_size=4, embed_dim=128, proj_dim=64)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

# Dummy data
batch_size = 8
images = torch.randn(batch_size, 3, 32, 32)
token_ids = torch.randint(0, 100, (batch_size, 16))

# Training step
result = train_step(model, images, token_ids, optimizer)
print(f"Training loss: {result['loss']:.4f}")
print(f"Temperature tau: {result['tau']:.4f}")

# Zero-shot classification with 5 "classes"
class_tokens = torch.randint(0, 100, (5, 16))
preds, probs = zero_shot_predict(model, images, class_tokens)
assert preds.shape == (batch_size,), f"Expected ({batch_size},), got {preds.shape}"
assert probs.shape == (batch_size, 5), f"Expected ({batch_size}, 5), got {probs.shape}"
assert torch.allclose(probs.sum(dim=1), torch.ones(batch_size), atol=1e-5)
print(f"Predictions shape: {preds.shape} -- CORRECT")
print(f"Probabilities shape: {probs.shape} -- CORRECT")
print(f"Probabilities sum to 1: {torch.allclose(probs.sum(1), torch.ones(batch_size))} -- CORRECT")
print("\nAll checks passed!")